In [ ]:
# 1st code

In [3]:
from google.colab import files
import pandas as pd

uploaded = files.upload()   # upload dialog আসবে

# uploaded ফাইলের নাম ধরো
filename = list(uploaded.keys())[0]

#df = pd.read_csv(filename)      # CSV হলে
df = pd.read_excel(filename)  # Excel হলে

df.head()


Saving ALL_FULL_DATA MODIFIED.xlsx to ALL_FULL_DATA MODIFIED (1).xlsx


,paper_name,url,headline,article,publication_date,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 555,Unnamed: 556,Unnamed: 557,Unnamed: 558,Unnamed: 559,Unnamed: 560,Unnamed: 561,Unnamed: 562,Unnamed: 563,Unnamed: 564
0,thedailystar,https://www.thedailystar.net/business/economy/...,BB probe into Union Bank: ‘S Alam staffer’ too...,"A ""staffer of S Alam Group"" took out Tk 118 cr...",2024-10-29 02:33:00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,thedailystar,https://www.thedailystar.net/business/economy/...,"Workers block highway for 2nd day, 15-km tailb...","Following the blockade, the traffic congestion...",2024-10-01 14:08:00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,thedailystar,https://www.thedailystar.net/business/news/ban...,Bangladesh holds just 0.01% of global potato m...,The thriving global potato industry is current...,2025-12-13 19:38:00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,thedailystar,https://www.thedailystar.net/business/economy/...,Farmers reduce potato acreage for next year af...,After suffering staggering losses from a bumpe...,2025-12-16 00:41:00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,thedailystar,https://www.thedailystar.net/business/organisa...,Investment Corporation of Bangladesh holds 49t...,The Investment Corporation of Bangladesh (ICB)...,2025-12-15 20:30:00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# ==============================
# 1. Import Libraries
# ==============================
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from scipy import sparse
import joblib


# ==============================
# 3. Remove Unnecessary Columns
# ==============================
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# ==============================
# 4. Handle Missing Values
# ==============================
df = df.dropna(subset=["article"])
df["headline"] = df["headline"].fillna("").astype(str)
df["article"] = df["article"].astype(str)

# Combine headline + article
df["text"] = (df["headline"] + ". " + df["article"]).str.strip()

# Date cleaning
df["publication_date"] = pd.to_datetime(df["publication_date"], errors="coerce")
df = df.dropna(subset=["publication_date"])

print("Rows after cleaning:", len(df))

# ==============================
# 5. Text Preprocessing
# ==============================
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)   # remove URLs
    text = re.sub(r"[^a-z0-9\s\.\,\%\$\-\']", " ", text)  # finance-friendly
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(preprocess_text)

# ==============================
# 6. Tokenization
# ==============================
token_pattern = re.compile(
    r"[a-z]+(?:'[a-z]+)?|[0-9]+(?:\.[0-9]+)?%?|\$[0-9]+(?:\.[0-9]+)?"
)

def tokenize(text):
    return token_pattern.findall(text)

df["tokens"] = df["clean_text"].apply(tokenize)

# ==============================
# 7. Lemmatization (Light Rule-Based)
# ==============================
STOPWORDS = set("""
a an and are as at be by for from has have he her his i in is it its of on or our
that the their them they this to was were will with you your
""".split())

def light_lemmatize(token):
    if token.endswith("ies") and len(token) > 4:
        return token[:-3] + "y"
    if token.endswith("ing") and len(token) > 5:
        return token[:-3]
    if token.endswith("ed") and len(token) > 4:
        return token[:-2]
    if token.endswith("es") and len(token) > 4:
        return token[:-2]
    if token.endswith("s") and len(token) > 3:
        return token[:-1]
    return token

def lemmatize_tokens(tokens):
    lemmas = []
    for t in tokens:
        if t in STOPWORDS or len(t) < 2:
            continue
        lemmas.append(light_lemmatize(t))
    return lemmas

df["lemm_tokens"] = df["tokens"].apply(lemmatize_tokens)
df["lemm_text"] = df["lemm_tokens"].apply(lambda x: " ".join(x))

# ==============================
# 8. Vectorization
# ==============================

# ---- Bag of Words ----
bow_vectorizer = CountVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2
)
X_bow = bow_vectorizer.fit_transform(df["lemm_text"])

# ---- TF-IDF ----
tfidf_vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2
)
X_tfidf = tfidf_vectorizer.fit_transform(df["lemm_text"])

print("BoW shape:", X_bow.shape)
print("TF-IDF shape:", X_tfidf.shape)

# ==============================
# 9. Save Outputs
# ==============================
df.to_csv("processed_news_dataset.csv", index=False)

joblib.dump(bow_vectorizer, "bow_vectorizer.pkl")
joblib.dump(tfidf_vectorizer, "tfidf_vectorizer.pkl")

sparse.save_npz("bow_matrix.npz", X_bow)
sparse.save_npz("tfidf_matrix.npz", X_tfidf)

print("All preprocessing steps completed successfully.")


Rows after cleaning: 8305
BoW shape: (8305, 8000)
TF-IDF shape: (8305, 8000)
All preprocessing steps completed successfully.


In [9]:
from google.colab import files

files.download("processed_news_dataset.csv")
files.download("tfidf_vectorizer.pkl")
files.download("tfidf_matrix.npz")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
files.download("bow_vectorizer.pkl")
files.download("bow_matrix.npz")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import joblib
from scipy import sparse

# Load vectorizer
tfidf_vectorizer = joblib.load("tfidf_vectorizer.pkl")

# Load matrix
X_tfidf = sparse.load_npz("tfidf_matrix.npz")

print(X_tfidf.shape)


(8305, 8000)


In [ ]:
# second code

In [1]:
!pip install pandas scikit-learn


In [2]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
filename = list(uploaded.keys())[0]

# CSV বা Excel handle
if filename.endswith(".csv"):
    df = pd.read_csv(filename)
else:
    df = pd.read_excel(filename)

print("Original Dataset")
display(df.head())
print("Shape:", df.shape)


Saving ALL_FULL_DATA MODIFIED.xlsx to ALL_FULL_DATA MODIFIED.xlsx
Original Dataset


,paper_name,url,headline,article,publication_date,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 555,Unnamed: 556,Unnamed: 557,Unnamed: 558,Unnamed: 559,Unnamed: 560,Unnamed: 561,Unnamed: 562,Unnamed: 563,Unnamed: 564
0,thedailystar,https://www.thedailystar.net/business/economy/...,BB probe into Union Bank: ‘S Alam staffer’ too...,"A ""staffer of S Alam Group"" took out Tk 118 cr...",2024-10-29 02:33:00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,thedailystar,https://www.thedailystar.net/business/economy/...,"Workers block highway for 2nd day, 15-km tailb...","Following the blockade, the traffic congestion...",2024-10-01 14:08:00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,thedailystar,https://www.thedailystar.net/business/news/ban...,Bangladesh holds just 0.01% of global potato m...,The thriving global potato industry is current...,2025-12-13 19:38:00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,thedailystar,https://www.thedailystar.net/business/economy/...,Farmers reduce potato acreage for next year af...,After suffering staggering losses from a bumpe...,2025-12-16 00:41:00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,thedailystar,https://www.thedailystar.net/business/organisa...,Investment Corporation of Bangladesh holds 49t...,The Investment Corporation of Bangladesh (ICB)...,2025-12-15 20:30:00,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Shape: (8334, 565)


In [3]:
# Remove unnamed columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

# Drop rows with missing article/text
df = df.dropna(subset=["article"])

# Fill headline if missing
df["headline"] = df["headline"].fillna("")

print("After Cleaning")
display(df.head())
print("Shape:", df.shape)


After Cleaning


,paper_name,url,headline,article,publication_date
0,thedailystar,https://www.thedailystar.net/business/economy/...,BB probe into Union Bank: ‘S Alam staffer’ too...,"A ""staffer of S Alam Group"" took out Tk 118 cr...",2024-10-29 02:33:00
1,thedailystar,https://www.thedailystar.net/business/economy/...,"Workers block highway for 2nd day, 15-km tailb...","Following the blockade, the traffic congestion...",2024-10-01 14:08:00
2,thedailystar,https://www.thedailystar.net/business/news/ban...,Bangladesh holds just 0.01% of global potato m...,The thriving global potato industry is current...,2025-12-13 19:38:00
3,thedailystar,https://www.thedailystar.net/business/economy/...,Farmers reduce potato acreage for next year af...,After suffering staggering losses from a bumpe...,2025-12-16 00:41:00
4,thedailystar,https://www.thedailystar.net/business/organisa...,Investment Corporation of Bangladesh holds 49t...,The Investment Corporation of Bangladesh (ICB)...,2025-12-15 20:30:00


Shape: (8305, 5)


In [4]:
df["text"] = (df["headline"] + ". " + df["article"]).str.strip()

print("After Integration (headline + article)")
display(df[["headline", "article", "text"]].head())


After Integration (headline + article)


,headline,article,text
0,BB probe into Union Bank: ‘S Alam staffer’ too...,"A ""staffer of S Alam Group"" took out Tk 118 cr...",BB probe into Union Bank: ‘S Alam staffer’ too...
1,"Workers block highway for 2nd day, 15-km tailb...","Following the blockade, the traffic congestion...","Workers block highway for 2nd day, 15-km tailb..."
2,Bangladesh holds just 0.01% of global potato m...,The thriving global potato industry is current...,Bangladesh holds just 0.01% of global potato m...
3,Farmers reduce potato acreage for next year af...,After suffering staggering losses from a bumpe...,Farmers reduce potato acreage for next year af...
4,Investment Corporation of Bangladesh holds 49t...,The Investment Corporation of Bangladesh (ICB)...,Investment Corporation of Bangladesh holds 49t...


In [5]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s\.\,\%\$\-\']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

print("After Transformation (clean_text)")
display(df[["text", "clean_text"]].head())


After Transformation (clean_text)


,text,clean_text
0,BB probe into Union Bank: ‘S Alam staffer’ too...,bb probe into union bank s alam staffer took t...
1,"Workers block highway for 2nd day, 15-km tailb...","workers block highway for 2nd day, 15-km tailb..."
2,Bangladesh holds just 0.01% of global potato m...,bangladesh holds just 0.01% of global potato m...
3,Farmers reduce potato acreage for next year af...,farmers reduce potato acreage for next year af...
4,Investment Corporation of Bangladesh holds 49t...,investment corporation of bangladesh holds 49t...


In [6]:
df = df[[
    "paper_name",
    "publication_date",
    "clean_text"
]]

print("After Reduction (selected columns)")
display(df.head())


After Reduction (selected columns)


,paper_name,publication_date,clean_text
0,thedailystar,2024-10-29 02:33:00,bb probe into union bank s alam staffer took t...
1,thedailystar,2024-10-01 14:08:00,"workers block highway for 2nd day, 15-km tailb..."
2,thedailystar,2025-12-13 19:38:00,bangladesh holds just 0.01% of global potato m...
3,thedailystar,2025-12-16 00:41:00,farmers reduce potato acreage for next year af...
4,thedailystar,2025-12-15 20:30:00,investment corporation of bangladesh holds 49t...


In [7]:
df["publication_date"] = pd.to_datetime(df["publication_date"], errors="coerce")
df = df.dropna(subset=["publication_date"])

df["year"] = df["publication_date"].dt.year
df["month"] = df["publication_date"].dt.month

print("After Discretization (year, month)")
display(df.head())


After Discretization (year, month)


,paper_name,publication_date,clean_text,year,month
0,thedailystar,2024-10-29 02:33:00,bb probe into union bank s alam staffer took t...,2024,10
1,thedailystar,2024-10-01 14:08:00,"workers block highway for 2nd day, 15-km tailb...",2024,10
2,thedailystar,2025-12-13 19:38:00,bangladesh holds just 0.01% of global potato m...,2025,12
3,thedailystar,2025-12-16 00:41:00,farmers reduce potato acreage for next year af...,2025,12
4,thedailystar,2025-12-15 20:30:00,investment corporation of bangladesh holds 49t...,2025,12


In [8]:
token_pattern = re.compile(
    r"[a-z]+(?:'[a-z]+)?|[0-9]+(?:\.[0-9]+)?%?|\$[0-9]+(?:\.[0-9]+)?"
)

def tokenize(text):
    return token_pattern.findall(text)

df["tokens"] = df["clean_text"].apply(tokenize)

print("After Tokenization")
display(df[["clean_text", "tokens"]].head())


After Tokenization


,clean_text,tokens
0,bb probe into union bank s alam staffer took t...,"[bb, probe, into, union, bank, s, alam, staffe..."
1,"workers block highway for 2nd day, 15-km tailb...","[workers, block, highway, for, 2, nd, day, 15,..."
2,bangladesh holds just 0.01% of global potato m...,"[bangladesh, holds, just, 0.01%, of, global, p..."
3,farmers reduce potato acreage for next year af...,"[farmers, reduce, potato, acreage, for, next, ..."
4,investment corporation of bangladesh holds 49t...,"[investment, corporation, of, bangladesh, hold..."


In [9]:
STOPWORDS = set("""
a an and are as at be by for from has have he her his i in is it its of on our
that the their them they this to was were will with you your
""".split())

def light_lemmatize(token):
    if token.endswith("ies") and len(token) > 4:
        return token[:-3] + "y"
    if token.endswith("ing") and len(token) > 5:
        return token[:-3]
    if token.endswith("ed") and len(token) > 4:
        return token[:-2]
    if token.endswith("es") and len(token) > 4:
        return token[:-2]
    if token.endswith("s") and len(token) > 3:
        return token[:-1]
    return token

def lemmatize(tokens):
    return [light_lemmatize(t) for t in tokens if t not in STOPWORDS and len(t) > 2]

df["lemm_tokens"] = df["tokens"].apply(lemmatize)
df["lemm_text"] = df["lemm_tokens"].apply(lambda x: " ".join(x))

print("After Lemmatization")
display(df[["tokens", "lemm_text"]].head())


After Lemmatization


,tokens,lemm_text
0,"[bb, probe, into, union, bank, s, alam, staffe...",probe into union bank alam staffer took 118 sa...
1,"[workers, block, highway, for, 2, nd, day, 15,...",worker block highway day tailback ashulia foll...
2,"[bangladesh, holds, just, 0.01%, of, global, p...",bangladesh hold just 0.01% global potato marke...
3,"[farmers, reduce, potato, acreage, for, next, ...",farmer reduce potato acreage next year after h...
4,"[investment, corporation, of, bangladesh, hold...",investment corporation bangladesh hold agm inv...


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    min_df=2
)

X_tfidf = tfidf.fit_transform(df["lemm_text"])

print("TF-IDF Matrix Shape:", X_tfidf.shape)


TF-IDF Matrix Shape: (8305, 5000)


In [11]:
df.to_csv("final_processed_dataset.csv", index=False)

import joblib
from scipy import sparse

joblib.dump(tfidf, "tfidf_vectorizer.pkl")
sparse.save_npz("tfidf_matrix.npz", X_tfidf)

print("Files saved successfully!")


Files saved successfully!


In [14]:
df.to_csv("final_processed_dataset.csv", index=False)


In [15]:
from google.colab import files
files.download("final_processed_dataset.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>